<a href="https://colab.research.google.com/github/RajendharAre/Innomatics_Research_Labs_IN226052302/blob/main/GenAI_NLP_Task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformers torch --quiet

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [4]:
# Define the pre-trained model name from Hugging Face Model Hub
MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"Loading model: {MODEL_NAME}")
print("This may take a moment on the first run...\n")

# Load the tokenizer — converts text to token IDs the model understands
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load the pre-trained causal language model weights
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

print("Model loaded successfully!")

Loading model: microsoft/DialoGPT-medium
This may take a moment on the first run...



config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully!


In [5]:
def generate_response(user_input, chat_history_ids=None):
    """
    Generate a chatbot response for the given user input.

    Parameters:
        user_input (str)        : The message typed by the user.
        chat_history_ids (tensor): Token IDs of the entire conversation so far.
                                   None on the first turn.

    Returns:
        response (str)          : The chatbot's reply as plain text.
        chat_history_ids (tensor): Updated conversation history (used in next turn).
    """

    # Encode the new user message and append the EOS token
    # EOS (End of Sequence) token tells the model where user input ends
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"   # return PyTorch tensor
    )

    # Append the new input to the existing conversation history
    # On the first turn, chat_history_ids is None, so we use just the new input
    bot_input_ids = (
        torch.cat([chat_history_ids, new_input_ids], dim=-1)
        if chat_history_ids is not None
        else new_input_ids
    )

    # Generate a response using the model
    # max_length        : cap total tokens (history + response)
    # do_sample         : enables sampling for more creative/varied responses
    # top_k             : consider only top 50 probable next tokens
    # top_p             : nucleus sampling — tokens covering top 90% probability mass
    # temperature       : controls randomness (lower = more focused)
    # repetition_penalty: discourages repeating the same phrases
    # pad_token_id      : use EOS token as padding token (required by DialoGPT)
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        do_sample=True,
        top_k=50,
        top_p=0.9,
        temperature=0.75,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode ONLY the newly generated response tokens (not the full history)
    # bot_input_ids.shape[-1] skips the prompt tokens we already sent in
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True   # remove EOS/PAD/BOS tokens from output
    )

    return response, chat_history_ids

In [7]:
def run_chatbot():
    """
    Main chatbot loop.
    Continuously reads user input, generates a response,
    and maintains conversation history until the user exits.
    """

    print("=" * 55)
    print("         AI Chatbot — Powered by DialoGPT")
    print("=" * 55)
    print('Type "exit" or "quit" to end the conversation.\n')
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("-" * 55)

    # Initialize conversation history as None (no history on first turn)
    chat_history_ids = None

    # ── Continuous conversation loop ────────────────────────────────────────
    while True:

        # Step 1: Accept user input
        user_input = input("You: ").strip()

        # Step 2: Check for empty input — prompt again without generating
        if not user_input:
            print("Chatbot: Please type something so I can help you!")
            continue

        # Step 3: Check exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("-" * 55)
            print("Chatbot: Goodbye! Have a great day. See you soon!")
            print("=" * 55)
            break

        # Step 4: Generate a response using the transformer model
        response, chat_history_ids = generate_response(user_input, chat_history_ids)

        # Step 5: Display the chatbot's response
        print(f"Chatbot: {response}")
        print("-" * 55)


# ── Entry point ─────────────────────────────────────────────────────────────
run_chatbot()

         AI Chatbot — Powered by DialoGPT
Type "exit" or "quit" to end the conversation.

Chatbot: Hello! I am your AI assistant. How can I help you today?
-------------------------------------------------------
You: Hi
Chatbot: I don't see a lot of people around here. You may have been the first, but we're still pretty active! We'd love to have you join us over at r kittapsalt!
-------------------------------------------------------
You: What is you favourite game?
Chatbot: The only question that matters : Skyrim or Skyrim 2? Or Elder Scrolls 6? Hint : It's not Skyrim
-------------------------------------------------------
You: exit
-------------------------------------------------------
Chatbot: Goodbye! Have a great day. See you soon!
